<a href="https://colab.research.google.com/github/smit-r/Shape-Generation-using-Spatially-Partitioned-Point-Clouds/blob/main/KDTree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python -m pip install open3d

In [2]:
import numpy as np
import scipy as scy
import os
import open3d
import numpy as np
import sklearn.decomposition

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/My Drive/Colab Notebooks/shapenet_chair_data/shapenet-chairs-pcd

/content/drive/My Drive/Colab Notebooks/shapenet_chair_data/shapenet-chairs-pcd


#### KDTree

In [4]:
## performs kdtree sorting on pointcloud
def kdtree(pointCloud, level):
  if (len(pointCloud)<=1):
    return pointCloud
  pointlist = np.array(pointCloud)
  plane = getPlane(level)
  median = getkthPosition(pointlist, plane, 0, len(pointlist)-1, int(len(pointlist)/2))
  leftArray = median[:int(len(median)/2)]
  rightArray = median[int((len(median)/2))+1:]
  leftSortedArray = kdtree(leftArray, level+1)
  rightSortedArray = kdtree(rightArray, level+1)
  return np.concatenate((leftSortedArray, [median[int(len(median)/2)]], rightSortedArray))

In [5]:
def getPlane(level):
  plane = level%3
  return plane

In [6]:
def getPartition(pointlist, axis, l, r):
  i = l
  x = pointlist[r, axis]
  for j in range(l, r):
    if pointlist[j,axis]<x:
      pointlist[[i,j]] = pointlist[[j,i]]
      i += 1
  pointlist[[i,r]] = pointlist[[r,i]]
  return i

In [ ]:
def getkthPosition(pointlist, axis, l, r, k):
  # quickselect: rearranges pointlist so the k-th smallest (along axis) is at index k
  if (l > r):
    return pointlist
  i = getPartition(pointlist, axis, l, r)
  if (i == k):
    return pointlist
  elif (i < k):
    return getkthPosition(pointlist, axis, i+1, r, k)
  else:
    return getkthPosition(pointlist, axis, l, i-1, k)

#### PCA Analysis

In [ ]:
## load the dataset
dataset = []
for f in os.listdir():
  if f[-4:]==".pcd":
    pcd = open3d.io.read_point_cloud(f)
    pointcloud = np.asarray(pcd.points)
    dataset.append(pointcloud)
  print(f)

In [ ]:
# sort using KDTree
data = np.array(dataset)
for i in range(len(dataset)):
  shape = data[i]
  shape = kdtree(shape, 0)
  data[i] = shape
  print(i)

In [ ]:
data = np.load('/content/drive/My Drive/shapenetdata.npy')

In [ ]:
# construct matrix P: one row per shape, points flattened as x0,y0,z0,x1,y1,z1,...
n_shapes = len(data)
P = np.reshape(data, [n_shapes, -1]).astype(np.float64) #(n_shapes, 3*n_points)
n_points = P.shape[1] // 3
mean = np.mean(P, axis=0) #(3*n_points, )
Pu = P - mean #P[i]-mean
Pu.shape

In [4]:
# PCA
def pcaAnalysis(Pu):
  pca = sklearn.decomposition.PCA(n_components=100)
  pca.fit(Pu)
  comp = pca.components_
  return comp

In [48]:
comp = pcaAnalysis(Pu)

In [49]:
# transform and reconstruct
Pcomp = np.dot(Pu, comp.T)
print(Pcomp.shape)
Precon = np.dot(Pcomp, comp)
print(Precon.shape)
errors = Pu - Precon
errors = errors*errors #calculate errors for each shape
print(errors.shape)
errors = np.sum(errors, axis=1) #(5000, )
print(errors.shape)
errlist = [errors]

(5000, 100)
(5000, 3000)
(5000, 3000)
(5000,)


In [ ]:
# sample two points randomly
def samplePoints():
  points = np.random.randint(0, n_points, 2)
  return points

In [24]:
# swap points
def swap(shape, points):
  shape[3*points[0]], shape[3*points[1]] = shape[3*points[1]], shape[3*points[0]] # swap x
  shape[3*points[0]+1], shape[3*points[1]+1] = shape[3*points[1]+1], shape[3*points[0]+1] #swap y
  shape[3*points[0]+2], shape[3*points[1]+2] = shape[3*points[1]+2], shape[3*points[0]+2] #swap z
  return shape

In [ ]:
# calculate error
def error(shape, shape_):
  err = shape - shape_
  err = np.dot(err.T, err)
  return err

# PCA reconstruction error of one mean-centred shape
def reconError(shape_centered, comp):
  recon = np.dot(np.dot(shape_centered, comp.T), comp)
  return error(shape_centered, recon)

In [ ]:
# ordering using PCA Reconstruction Error
# as given in paper
# Points are swapped in the original shape (P), which is then centred with the
# dataset mean to measure its PCA reconstruction error. A swap is kept only if it
# lowers the shape's *current* error, so the error can only go down.
save_path = '/content/drive/My Drive/'
k = 10**4
count = 0
mean = np.mean(P, axis=0)
for s in range(len(P)):
  cur = np.array(P[s])
  cur_err = reconError(cur - mean, comp)
  for i in range(k):
    cand = swap(np.array(cur), samplePoints())
    cand_err = reconError(cand - mean, comp)
    if cand_err < cur_err:
      cur, cur_err = cand, cand_err
      count += 1
  P[s] = cur
  print(count, s)
  # refit PCA on the updated shapes before moving on to the next shape
  mean = np.mean(P, axis=0)
  Pu = P - mean
  comp = pcaAnalysis(Pu)
  if s%50==0 or s==len(P)-1: #calculate errors for each shape and store in errlist
    Pcomp = np.dot(Pu, comp.T)
    Precon = np.dot(Pcomp, comp)
    errors = Pu - Precon
    errors = errors*errors
    errors = np.sum(errors, axis=1)
    errlist.append(errors)
    np.save(save_path+'errlist', np.array(errlist))
    np.save(save_path+'Pu', P)

## results after performing swaping for 1000 shapes

[Reconstruction Error](https://drive.google.com/file/d/1MGAZNpFGKdh0b-mEuOIDiWvut3WGs_vA/view?usp=sharing)

In [ ]:
def swapAll(P_, points):
  for i in range(len(points)):
    P_[i] = swap(P_[i], points[i])
  return P_

In [ ]:
def sampleAll():
  points = np.random.randint(0, n_points, (len(P), 2))
  return points

In [28]:
def errAll(P_, comp):
  mean = np.mean(P_, axis=0)
  Pu_ = P_ - mean
  Pcomp = np.dot(Pu_, comp.T)
  Precon = np.dot(Pcomp, comp)
  errors = Pu_ - Precon
  errors = errors*errors
  errors = np.sum(errors, axis=1)
  return errors

In [ ]:
#### optimized training loop
# Every round proposes one random swap per shape for all shapes at once and keeps
# the swaps that lower that shape's current error. PCA is refitted after
# swap_rounds rounds.
save_path = '/content/drive/My Drive/'
n_iters = 1000      # number of PCA refits
swap_rounds = 40    # swap rounds between PCA refits
errors = errAll(P, comp)
for it in range(n_iters):
  for _ in range(swap_rounds):
    P_ = swapAll(np.array(P), sampleAll())
    # calculate error
    err = errAll(P_, comp)
    # compare: keep the swap only where it lowered the error
    better = err < errors
    P = np.where(better[:, None], P_, P)
    errors = np.where(better, err, errors)
  Pu = P - np.mean(P, axis=0)
  comp = pcaAnalysis(Pu)
  errors = errAll(P, comp)
  errlist.append(errors)
  np.save(save_path+'vec_errlist1', np.array(errlist))
  np.save(save_path+'vec_Pu1', P)
  print(it, np.sum(errors))

## results after 400 iterations

[Reconstruction Error](https://drive.google.com/file/d/1-46E_roxPaFlTNoULcBjBYmD3uCfdoRH/view?usp=sharing)

#### GAN 

In [57]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers
import numpy as np 
import matplotlib.pyplot as plt 
from tensorflow.keras.constraints import Constraint
from tensorflow.keras import backend

In [58]:
class ClipConstraint(Constraint):
	# set clip value when initialized
	def __init__(self, clip_value):
		self.clip_value = clip_value
 
	# clip model weights to hypercube
	def __call__(self, weights):
		return backend.clip(weights, -self.clip_value, self.clip_value)
 
	# get the config
	def get_config(self):
		return {'clip_value': self.clip_value}

In [59]:
const = ClipConstraint(0.01)

In [60]:
latent_dim = 100

Generator = keras.Sequential(
    [keras.Input(shape=(latent_dim,)),
     layers.Dense(100),
     layers.BatchNormalization(),
     layers.ReLU(),
     layers.Dense(100),
     layers.BatchNormalization(),
     layers.ReLU(),
     layers.Dense(100),
     layers.BatchNormalization(),
     layers.ReLU(),
     layers.Dense(100),
     layers.BatchNormalization(),
     layers.ReLU(),
     layers.Dense(100)
    ],
    name = 'Generator'
)

In [ ]:
Discriminator = keras.Sequential(
    [keras.Input(shape=(latent_dim,)),
     layers.Dense(100, kernel_constraint=const),
     layers.LeakyReLU(alpha=0.2),
     layers.Dense(100, kernel_constraint=const),
     layers.LeakyReLU(alpha=0.2),
     layers.Dense(100, kernel_constraint=const),
     layers.LeakyReLU(alpha=0.2),
     layers.Dense(100, kernel_constraint=const),
     layers.LeakyReLU(alpha=0.2),
     layers.Dense(1, kernel_constraint=const)
    ],
    name = 'Discriminator'
)

In [ ]:
# Wasserstein loss. Convention: the critic scores real shapes low (label +1) and
# generated shapes high (label -1); the generator tries to make its scores low.
def wass_loss_fn(label, coeff):
  wgloss = tf.reduce_mean(coeff*label)
  return wgloss

In [ ]:
# optional moment-matching term: squared difference of the batch means
def loss_fn(real, fake):
  Eloss = tf.reduce_mean(real, axis = 0) - tf.reduce_mean(fake, axis = 0)
  Sloss = tf.reduce_sum(tf.square(Eloss))
  return Sloss

In [ ]:
## training loop (WGAN with weight clipping, Arjovsky et al. 2017)

gen_opt = keras.optimizers.RMSprop(learning_rate=5e-5)
dis_opt = keras.optimizers.RMSprop(learning_rate=5e-5)

n_critic = 5     # critic updates per generator update
mm_weight = 0.0  # weight of the optional moment-matching term in the generator loss

gloss = []
dlossr = []
dlossf = []

@tf.function
def critic_step(real_coeff):
  batch_size = tf.shape(real_coeff)[0]
  noise = tf.random.uniform(shape=[batch_size, latent_dim], minval=-1, maxval=1)
  with tf.GradientTape() as tape:
    fake_coeff = Generator(noise, training=True)
    d_loss_fake = wass_loss_fn(-1.0, Discriminator(fake_coeff, training=True))
    d_loss_real = wass_loss_fn(1.0, Discriminator(real_coeff, training=True))
    d_loss = d_loss_real + d_loss_fake
  grads = tape.gradient(d_loss, Discriminator.trainable_weights)
  dis_opt.apply_gradients(zip(grads, Discriminator.trainable_weights))
  return d_loss_real, d_loss_fake

@tf.function
def generator_step(real_coeff):
  batch_size = tf.shape(real_coeff)[0]
  noise = tf.random.uniform(shape=[batch_size, latent_dim], minval=-1, maxval=1)
  with tf.GradientTape() as tape:
    gen_coeff = Generator(noise, training=True)
    g_loss = wass_loss_fn(1.0, Discriminator(gen_coeff, training=True))
    g_loss += mm_weight * loss_fn(real_coeff, gen_coeff)
  grads = tape.gradient(g_loss, Generator.trainable_weights)
  gen_opt.apply_gradients(zip(grads, Generator.trainable_weights))
  return g_loss

In [ ]:
# The GAN learns the PCA coefficients of the reordered shapes. The basis (comp)
# and mean used here are the same ones used later to turn generated
# coefficients back into points. Coefficients are scaled to unit variance per
# component, which suits the critic's weight clipping.
Pmean = np.mean(P, axis=0)
comp = pcaAnalysis(P - Pmean)
Pcomp = np.dot(P - Pmean, comp.T)
coeff_std = Pcomp.std(axis=0)
train_coeff = (Pcomp / coeff_std).astype("float32")

In [ ]:
## set batch_size and epochs here and run this cell for training

batch_size = 64
dataset = tf.data.Dataset.from_tensor_slices(train_coeff)
dataset = dataset.shuffle(buffer_size=1024).batch(batch_size, drop_remainder=True)


epochs = 500
save_dir = ''

step_count = 0
for epoch in range(epochs):
  print("\nstart epoch", epoch)
  for step, batch in enumerate(dataset):
    # critic update every step, generator update every n_critic steps
    d_loss_real, d_loss_fake = critic_step(batch)
    dlossr.append(float(d_loss_real))
    dlossf.append(float(d_loss_fake))
    if step_count % n_critic == 0:
      g_loss = generator_step(batch)
      gloss.append(float(g_loss))
    step_count += 1

    #logging
    if step%50==0:
      print("discriminator loss at step %d:  d_loss_real : %.4f , d_loss_fake : %.4f" % (step, d_loss_real, d_loss_fake))
      print("generator loss at step %d: %.4f" % (step, gloss[-1]))


      # save weights and gen_outputs

In [ ]:
plt.plot(np.arange(len(gloss)), gloss)

In [ ]:
plt.plot(np.arange(len(dlossf)), dlossf)

In [ ]:
plt.plot(np.arange(len(dlossr)), dlossr)

In [ ]:
gen_coeff = Generator(tf.random.uniform([1, latent_dim], -1, 1), training=False)
gen_coeff

In [ ]:
gen_coeff = np.array(gen_coeff[0]) * coeff_std  # undo the per-component scaling

In [ ]:
gen_shape = np.dot(gen_coeff, comp)
gen_shape = gen_shape + Pmean

In [93]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

In [ ]:
pc0 = gen_shape
x, y, z = np.reshape(pc0, (-1, 3)).T

In [96]:
ax.scatter(x,z,y)

In [ ]:
fig

In [5]:
pc = np.load("/content/drive/My Drive/vec_Pu1.npy")